<a href="https://colab.research.google.com/github/amairagoel/Audio-to-Synced-Translated-Lyrics/blob/main/service.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import httpx
from openai import OpenAI
import os

openai_client = OpenAI()

LRCLIB_URL = "https://lrclib.net/api/get"

async def fetch_lrclib_synced(artist: str, title: str) -> str:
    """Fetches raw timed LRC string arrays from open-source registries."""
    headers = {"User-Agent": "EnterpriseLyricsB2B/2.0.0 (contact@yourcompany.com)"}
    params = {"artist_name": artist.strip().lower(), "track_name": title.strip().lower()}

    async with httpx.AsyncClient() as client:
        try:
            response = await client.get(LRCLIB_URL, params=params, headers=headers, timeout=8.0)
            if response.status_code == 200:
                return response.json().get("syncedLyrics") or ""
            return ""
        except Exception:
            return ""

def translate_lrc_contextually(lrc_text: str) -> str:
    """Translates localized slang while strictly preserving original timestamps."""
    if not lrc_text.strip():
        return ""

    system_instruction = (
        "You are an enterprise-grade music localization translation system.\n"
        "Translate the following raw timestamped LRC lyrics contextually into English.\n"
        "Ensure regional urban street slang and idioms (especially Puerto Rican reggaeton expressions) "
        "are parsed conceptually rather than literally.\n"
        "CRITICAL: Do not alter, modify, or remove any timestamps (e.g., [00:12.34]). Leave them exactly intact.\n"
        "Output ONLY the translated LRC strings. Do not add metadata headers or markdown backticks."
    )

    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_instruction},
            {"role": "user", "content": lrc_text}
        ],
        temperature=0.2
    )
    return response.choices[0].message.content.strip()

async def dispatch_webhook_callback(callback_url: str, payload: dict):
    """Asynchronously returns completed payloads back to the client destination URI."""
    async with httpx.AsyncClient() as client:
        try:
            # We enforce a retry buffer or timeout configuration to keep workers stable
            await client.post(callback_url, json=payload, timeout=15.0)
            print(f"[Webhook Log] Callback successfully sent to {callback_url}")
        except Exception as e:
            print(f"[Webhook Error] Failed dispatch to {callback_url}: {str(e)}")